# Combat transitions → feature table (Colab tip #1)

**Track:** Colab GPU parallel — feature extract only. Does **not** train Critic/DT, does **not** touch game comms / EP bridge / `sts2.dll`.

**Input:** `transitions.npz` with keys `obs`, `next_obs`, `action`, `reward`, `done`, `action_mask`.

| Field | Shape / dtype |
|-------|----------------|
| `obs`, `next_obs` | `(N, 181)` float32 — combat obs_v1 |
| `action` | `(N,)` int — gym combat action index |
| `reward` | `(N,)` float32 |
| `done` | `(N,)` bool |
| `action_mask` | `(N, M)` — legal action mask (M ≈ action space) |

**Gate:** export ≥1000 rows; **no NaN/Inf** in numeric columns.

**Optional (not gated here):** HOLD replay JSONL `hold_turn_replay_*.jsonl` for hp/enemies_hp context in later tips.

In [ ]:
# Run once: clone repo for extract logic (or upload sts2_env/colab/ and fix sys.path)
# !git clone --depth 1 https://github.com/EienteiPharma/sts2-rl-agent.git /content/sts2-rl-agent

!pip install -q numpy pandas pyarrow

import sys
from pathlib import Path

REPO_ROOT = Path("/content/sts2-rl-agent")
if not REPO_ROOT.is_dir():
    raise SystemExit(
        "Clone sts2-rl-agent to /content/sts2-rl-agent (uncomment git clone) "
        "or upload combat_transition_features.py and add parent to sys.path."
    )
sys.path.insert(0, str(REPO_ROOT))

from sts2_env.colab.combat_transition_features import extract_combat_features, COMBAT_OBS_DIM

print("COMBAT_OBS_DIM", COMBAT_OBS_DIM)

In [ ]:
# --- Path config (do not hardcode /workspace) ---
from google.colab import files  # type: ignore

USE_UPLOAD = True  # False if you mounted Drive and set NPZ_PATH manually
NPZ_PATH = "/content/transitions.npz"
OUT_PATH = "/content/combat_features.parquet"  # or .npz
MIN_ROWS = 1000
SAMPLE_ROWS = 1000
SEED = 0

if USE_UPLOAD:
    uploaded = files.upload()  # pick transitions.npz in browser
    assert uploaded, "upload transitions.npz"
    name = next(iter(uploaded))
    with open(NPZ_PATH, "wb") as f:
        f.write(uploaded[name])
    print("wrote", NPZ_PATH, "from", name)

# Google Drive example (comment upload block above):
# from google.colab import drive
# drive.mount('/content/drive')
# NPZ_PATH = "/content/drive/MyDrive/sts2/transitions.npz"
# OUT_PATH = "/content/drive/MyDrive/sts2/combat_features.parquet"

In [ ]:
meta = extract_combat_features(
    NPZ_PATH,
    OUT_PATH,
    min_rows=MIN_ROWS,
    sample_rows=SAMPLE_ROWS,
    seed=SEED,
)
meta

In [ ]:
import numpy as np

assert meta["n_rows"] >= MIN_ROWS
assert meta["obs_shape"][1] == 181
print("OK gate: rows", meta["n_rows"], "obs_shape", meta["obs_shape"])

if OUT_PATH.endswith(".npz"):
    with np.load(OUT_PATH) as z:
        for k in z.files:
            a = z[k]
            if np.issubdtype(a.dtype, np.floating):
                assert np.isfinite(a).all(), k
else:
    import pandas as pd
    df = pd.read_parquet(OUT_PATH)
    num = df.select_dtypes(include=["float", "float32", "float64"])
    assert num.isna().sum().sum() == 0
    print("parquet columns", len(df.columns), "rows", len(df))
    df.head()

## Download artifact

Run the next cell to download `combat_features.parquet` to your laptop for tip #2 (not opened by this tip).

In [ ]:
from google.colab import files  # type: ignore
files.download(OUT_PATH)